In [0]:
from pyspark.sql import functions as F

print("Bronze layer notebook started")

Bronze layer notebook started


In [0]:
storage_account = "healthcarestoragerev01"

storage_key = dbutils.secrets.get(
    scope="healthcare-scope",
    key="storage-account-key"
)

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key
)

print("Storage authentication configured")

Storage authentication configured


In [0]:
# ============================================
# CELL 3: Define Landing and Bronze paths
# ============================================

storage_account = "healthcarestoragerev01"

input_container = "input"

landing_path = (
    f"abfss://{input_container}@"
    f"{storage_account}.dfs.core.windows.net"
)

bronze_path = (
    f"abfss://{input_container}@"
    f"{storage_account}.dfs.core.windows.net/bronze"
)

print("Landing path:")
print(landing_path)

print("Bronze path:")
print(bronze_path)

Landing path:
abfss://input@healthcarestoragerev01.dfs.core.windows.net
Bronze path:
abfss://input@healthcarestoragerev01.dfs.core.windows.net/bronze


In [0]:
# ============================================
# CELL 4: Check files in Landing/Input
# ============================================

display(
    dbutils.fs.ls(landing_path)
)

path,name,size,modificationTime
abfss://input@healthcarestoragerev01.dfs.core.windows.net/bronze/,bronze/,0,1788278227000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/departments.csv,departments.csv,502,1788276466000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/encounters.csv,encounters.csv,1080195,1788276458000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/gold/,gold/,0,1788282652000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/insurance_claim_data.csv,insurance_claim_data.csv,2416070,1788276476000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/patients.csv,patients.csv,765406,1788276458000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/providers.csv,providers.csv,1780,1788276452000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/quarantine/,quarantine/,0,1788281704000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/silver/,silver/,0,1788280123000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/transactions.csv,transactions.csv,2635842,1788276458000


In [0]:
# ============================================
# CELL 5: Define input CSV files
# ============================================

files = [
    "departments.csv",
    "encounters.csv",
    "insurance_claim_data.csv",
    "patients.csv",
    "providers.csv",
    "transactions.csv"
]

print("Files to process:")
for file in files:
    print(file)

Files to process:
departments.csv
encounters.csv
insurance_claim_data.csv
patients.csv
providers.csv
transactions.csv


In [0]:
# ============================================
# CELL 6: Test read departments.csv
# ============================================

file_path = f"{landing_path}/departments.csv"

df_departments = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(file_path)
)

print("departments.csv read successfully")

print("Record count:", df_departments.count())

display(df_departments.limit(5))

departments.csv read successfully
Record count: 20


DeptID,Name
DEPT001,Emergency
DEPT002,Cardiology
DEPT003,Neurology
DEPT004,Oncology
DEPT005,Pediatrics


In [0]:
# ============================================
# CELL 7: Read all Landing CSV files
# ============================================

dataframes = {}

for file in files:

    file_path = f"{landing_path}/{file}"

    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(file_path)
    )

    table_name = file.replace(".csv", "")

    dataframes[table_name] = df

    print(
        f"{file} -> {df.count()} records"
    )

print("All Landing files read successfully")

departments.csv -> 20 records
encounters.csv -> 10000 records
insurance_claim_data.csv -> 10000 records
patients.csv -> 5000 records
providers.csv -> 25 records
transactions.csv -> 10000 records
All Landing files read successfully


In [0]:
# ============================================
# CELL 8: Create Bronze folder
# ============================================

dbutils.fs.mkdirs(bronze_path)

print("Bronze folder created/verified")

Bronze folder created/verified


In [0]:
# ============================================
# CELL 9: Write departments to Bronze
# ============================================

df_departments = dataframes["departments"]

departments_bronze_path = f"{bronze_path}/departments"

(
    df_departments.write
    .format("delta")
    .mode("overwrite")
    .save(departments_bronze_path)
)

print("departments written to Bronze successfully")

departments written to Bronze successfully


In [0]:
# ============================================
# CELL 10: Verify departments Bronze data
# ============================================

display(
    dbutils.fs.ls(departments_bronze_path)
)

path,name,size,modificationTime
abfss://input@healthcarestoragerev01.dfs.core.windows.net/bronze/departments/_delta_log/,_delta_log/,0,1788278227000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/bronze/departments/part-00000-1727ec10-8a35-4d2f-bc1a-7e6942042d27-c000.snappy.parquet,part-00000-1727ec10-8a35-4d2f-bc1a-7e6942042d27-c000.snappy.parquet,1505,1788291555000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/bronze/departments/part-00000-36c5b796-5357-4b6a-8e5c-d0f6a589432b-c000.snappy.parquet,part-00000-36c5b796-5357-4b6a-8e5c-d0f6a589432b-c000.snappy.parquet,1450,1788290537000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/bronze/departments/part-00000-83b3fb5c-5c57-4fc6-af28-5e20f6b789e9-c000.snappy.parquet,part-00000-83b3fb5c-5c57-4fc6-af28-5e20f6b789e9-c000.snappy.parquet,1450,1788291839000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/bronze/departments/part-00000-a5f62e32-d501-4f99-9b63-f9e7b156c1a1-c000.snappy.parquet,part-00000-a5f62e32-d501-4f99-9b63-f9e7b156c1a1-c000.snappy.parquet,1450,1788290552000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/bronze/departments/part-00000-a8c7c32d-71c7-44ed-804c-a4f077ec2111-c000.snappy.parquet,part-00000-a8c7c32d-71c7-44ed-804c-a4f077ec2111-c000.snappy.parquet,1450,1788279209000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/bronze/departments/part-00000-ba5f5d5f-ba98-4410-a923-03a241ee1393-c000.snappy.parquet,part-00000-ba5f5d5f-ba98-4410-a923-03a241ee1393-c000.snappy.parquet,1450,1788447645000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/bronze/departments/part-00000-be2d8d40-4f9b-49ea-b983-112ae1db6d36-c000.snappy.parquet,part-00000-be2d8d40-4f9b-49ea-b983-112ae1db6d36-c000.snappy.parquet,1505,1788291548000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/bronze/departments/part-00000-c13db81c-e735-4428-a918-6779480dd5ce-c000.snappy.parquet,part-00000-c13db81c-e735-4428-a918-6779480dd5ce-c000.snappy.parquet,1450,1788279307000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/bronze/departments/part-00000-e79ca5c7-d04c-437f-b240-b0b09395c00b-c000.snappy.parquet,part-00000-e79ca5c7-d04c-437f-b240-b0b09395c00b-c000.snappy.parquet,1450,1788291832000


In [0]:
# ============================================
# CELL 11: Read Bronze Delta data
# ============================================

bronze_departments_df = (
    spark.read
    .format("delta")
    .load(departments_bronze_path)
)

print(
    "Bronze departments records:",
    bronze_departments_df.count()
)

display(bronze_departments_df.limit(5))

Bronze departments records: 20


DeptID,Name,_bronze_loaded_at
DEPT001,Emergency,null
DEPT002,Cardiology,null
DEPT003,Neurology,null
DEPT004,Oncology,null
DEPT005,Pediatrics,null


In [0]:
# ============================================
# CELL 12: Write all files to Bronze
# ============================================

for table_name, df in dataframes.items():

    output_path = f"{bronze_path}/{table_name}"

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .save(output_path)
    )

    print(
        f"Bronze completed: {table_name}"
    )

print("================================")
print("ALL FILES WRITTEN TO BRONZE")
print("================================")

Bronze completed: departments
Bronze completed: encounters
Bronze completed: insurance_claim_data
Bronze completed: patients
Bronze completed: providers
Bronze completed: transactions
ALL FILES WRITTEN TO BRONZE


In [0]:
# ============================================
# CELL 13: Verify Bronze layer
# ============================================

display(
    dbutils.fs.ls(bronze_path)
)

path,name,size,modificationTime
abfss://input@healthcarestoragerev01.dfs.core.windows.net/bronze/departments/,departments/,0,1788278227000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/bronze/encounters/,encounters/,0,1788278247000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/bronze/insurance_claim_data/,insurance_claim_data/,0,1788278253000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/bronze/patients/,patients/,0,1788278259000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/bronze/providers/,providers/,0,1788278264000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/bronze/transactions/,transactions/,0,1788278268000
